# 4. Dynamic Programming

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 4: Dynamic Programming** (Sayfa 73-92)

## İçindekiler
1. DP'ye Giriş *(s. 73-74)*
2. Policy Evaluation *(s. 74-76)*
3. Policy Improvement *(s. 76-78)*
4. Policy Iteration *(s. 80-81)*
5. Value Iteration *(s. 82-84)*
6. Gambler's Problem *(Example 4.3, s. 84)*

---
## 4.1 Dynamic Programming Nedir?

📖 **Referans:** Sutton & Barto, Sayfa 73-74

> *"The term dynamic programming (DP) refers to a collection of algorithms that can be used to compute optimal policies given a perfect model of the environment as a Markov decision process."* (s. 73)

**Dynamic Programming (DP)**, optimal policy hesaplamak için kullanılan bir algoritma ailesidir.

### DP'nin Gereksinimleri (s. 73)
- Environment'ın **tam modeli** bilinmeli: $p(s', r | s, a)$
- Bu nedenle DP **model-based** bir yöntemdir

> *"DP algorithms are obtained by turning Bellman equations... into update rules for improving approximations of the desired value functions."* (s. 73)

### DP'nin Avantajları (s. 73-74)
- Matematiksel olarak **optimal** çözüm garantisi
- Diğer RL metodlarının **temelini** oluşturur

### DP'nin Dezavantajları
- Model bilinmeli (çoğu zaman bilinmez)
- Büyük state space'lerde **computationally expensive** (s. 86-87)

In [ ]:
# Kod Örneği: Grid World Environment
# Referans: Sutton & Barto, Example 4.1 (s. 76-77) - 4x4 Gridworld

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

class GridWorld:
    """
    4x4 Grid World for DP examples.
    Referans: Example 4.1 (s. 76-77)
    
    "Consider the 4×4 gridworld shown below... The nonterminal states are 
    S = {1, 2, ..., 14}. There are four actions possible in each state, 
    A = {up, down, right, left}" (s. 76)
    """
    
    def __init__(self):
        self.rows = 4
        self.cols = 4
        self.n_states = 16
        self.n_actions = 4  # up, right, down, left
        self.terminal_states = [0, 15]  # Köşeler terminal
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        self.action_symbols = ['↑', '→', '↓', '←']
    
    def state_to_pos(self, s):
        return s // self.cols, s % self.cols
    
    def pos_to_state(self, row, col):
        return row * self.cols + col
    
    def step(self, state, action):
        """
        Returns (next_state, reward, done).
        "Actions that would take the agent off the grid leave the state unchanged." (s. 76)
        """
        if state in self.terminal_states:
            return state, 0, True
        
        row, col = self.state_to_pos(state)
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        
        next_state = self.pos_to_state(new_row, new_col)
        reward = -1  # "The reward is −1 on all transitions" (s. 76)
        done = next_state in self.terminal_states
        
        return next_state, reward, done
    
    def get_transitions(self, state, action):
        """
        Returns list of (probability, next_state, reward, done).
        Deterministic environment: p(s',r|s,a) = 1 for one outcome
        """
        next_state, reward, done = self.step(state, action)
        return [(1.0, next_state, reward, done)]

env = GridWorld()
print(f"Grid: {env.rows}x{env.cols}")
print(f"Terminal states: {env.terminal_states}")
print(f"Actions: {env.action_symbols}")

In [ ]:
def plot_values(env, V, title="State Values"):
    """Value function'ı görselleştir."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    V_grid = V.reshape(env.rows, env.cols)
    
    im = ax.imshow(V_grid, cmap='RdYlGn')
    
    for i in range(env.rows):
        for j in range(env.cols):
            state = env.pos_to_state(i, j)
            color = 'white' if abs(V_grid[i, j]) > 7 else 'black'
            ax.text(j, i, f'{V_grid[i, j]:.1f}', ha='center', va='center', 
                   fontsize=16, color=color, fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=14)
    plt.colorbar(im)
    plt.show()

def plot_policy(env, policy, V=None, title="Policy"):
    """Policy'yi görselleştir."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for s in range(env.n_states):
        row, col = env.state_to_pos(s)
        
        if s in env.terminal_states:
            color = 'lightgreen'
            text = 'T'
        else:
            color = 'white'
            text = env.action_symbols[policy[s]]
        
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                              facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        ax.text(col + 0.5, env.rows - row - 0.5, text,
               ha='center', va='center', fontsize=24, fontweight='bold')
        
        if V is not None:
            ax.text(col + 0.1, env.rows - row - 0.1, f'{V[s]:.1f}',
                   ha='left', va='top', fontsize=10, color='gray')
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

---
## 4.2 Policy Evaluation (Prediction)

📖 **Referans:** Sutton & Barto, Sayfa 74-76, Section 4.1

> *"First we consider how to compute the state-value function $v_\pi$ for an arbitrary policy π. This is called policy evaluation in the DP literature."* (s. 74)

Verilen bir policy $\pi$ için value function $V^\pi$ hesapla.

### Iterative Policy Evaluation - Equation 4.5 (s. 75)

Bellman expectation equation'ı **iteratif** olarak uygula:

$$v_{k+1}(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma v_k(s')]$$

> *"The sequence {$v_k$} can be shown in general to converge to $v_\pi$ as $k → ∞$"* (s. 75)

### Convergence Kriteri (s. 75)
Algorithm, $\max_s |v_{k+1}(s) - v_k(s)| < \theta$ olduğunda durur.

In [ ]:
def policy_evaluation(env, policy, gamma=1.0, theta=1e-8):
    """
    Iterative Policy Evaluation.
    Referans: Sutton & Barto, Algorithm (s. 75)
    
    "Input π, the policy to be evaluated"
    "Output V ≈ v_π"
    
    Args:
        env: Environment
        policy: Array of shape [n_states], action for each state
        gamma: Discount factor
        theta: Convergence threshold ("a small positive number")
    
    Returns:
        V: Value function array
    """
    # "Initialize V(s), for all s ∈ S+, arbitrarily except that V(terminal) = 0"
    V = np.zeros(env.n_states)
    
    iteration = 0
    history = [V.copy()]
    
    # "Repeat" loop (s. 75)
    while True:
        delta = 0  # "Δ ← 0"
        
        # "For each s ∈ S:"
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]  # "v ← V(s)"
            a = policy[s]
            
            # "V(s) ← Σ_a π(a|s) Σ_{s',r} p(s',r|s,a)[r + γV(s')]" - Eq 4.5
            new_v = 0
            for prob, next_s, reward, done in env.get_transitions(s, a):
                new_v += prob * (reward + gamma * V[next_s])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))  # "Δ ← max(Δ, |v − V(s)|)"
        
        iteration += 1
        history.append(V.copy())
        
        # "until Δ < θ"
        if delta < theta:
            break
    
    print(f"Policy Evaluation converged in {iteration} iterations")
    return V, history

In [ ]:
# Uniform Random Policy Evaluation
# Referans: Example 4.1 (s. 76-77) - "under the equiprobable random policy"

def evaluate_uniform_policy(env, gamma=1.0, theta=1e-8):
    """
    Uniform random policy evaluation.
    Referans: Example 4.1 (s. 76-77)
    
    "The left side of each cell is the value of the state; the right 
    side of each cell shows the policy...The center column shows the 
    sequence of approximations" (s. 77)
    """
    V = np.zeros(env.n_states)
    action_prob = 1.0 / env.n_actions  # "equiprobable random policy, π(up|·) = π(down|·) = π(right|·) = π(left|·) = 0.25"
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]
            new_v = 0
            
            # Sum over all actions with equal probability
            for a in range(env.n_actions):
                for prob, next_s, reward, done in env.get_transitions(s, a):
                    new_v += action_prob * prob * (reward + gamma * V[next_s])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        if delta < theta:
            break
    
    print(f"Converged in {iteration} iterations")
    return V

V_uniform = evaluate_uniform_policy(env)
plot_values(env, V_uniform, "V(s) for Uniform Random Policy (Example 4.1, s. 77)")

---
## 4.3 Policy Improvement

📖 **Referans:** Sutton & Barto, Sayfa 76-78, Section 4.2

> *"Our reason for computing the value function for a policy is to help find better policies."* (s. 76)

Mevcut value function $V^\pi$'den **daha iyi** bir policy $\pi'$ oluştur.

### Policy Improvement Theorem (s. 78)

> *"Let π and π' be any pair of deterministic policies such that, for all s ∈ S, $q_\pi(s, \pi'(s)) \geq v_\pi(s)$. Then the policy π' must be as good as, or better than, π."*

Eğer tüm $s$ için:
$$q_\pi(s, \pi'(s)) \geq v_\pi(s)$$

O zaman:
$$v_{\pi'}(s) \geq v_\pi(s)$$

### Greedy Policy - Equation 4.9 (s. 78)

$$\pi'(s) = \arg\max_a q_\pi(s, a) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_\pi(s')]$$

> *"The process of making a new policy that improves on an original policy, by making it greedy with respect to the value function of the original policy, is called policy improvement."* (s. 78)

In [ ]:
def policy_improvement(env, V, gamma=1.0):
    """
    V'ye göre greedy policy oluştur.
    Referans: Equation 4.9 (s. 78)
    
    "π'(s) = argmax_a Σ_{s',r} p(s',r|s,a)[r + γV(s')]"
    
    Returns:
        policy: Improved policy
    """
    policy = np.zeros(env.n_states, dtype=int)
    
    for s in range(env.n_states):
        if s in env.terminal_states:
            continue
        
        # Her action için Q(s,a) hesapla - Equation 4.9
        q_values = np.zeros(env.n_actions)
        
        for a in range(env.n_actions):
            for prob, next_s, reward, done in env.get_transitions(s, a):
                q_values[a] += prob * (reward + gamma * V[next_s])
        
        # Greedy action: argmax_a Q(s,a)
        policy[s] = np.argmax(q_values)
    
    return policy

# Uniform policy'nin V'sine göre improve et
improved_policy = policy_improvement(env, V_uniform)
plot_policy(env, improved_policy, V_uniform, "Improved Policy (from uniform) - Policy Improvement Theorem, s. 78")

---
## 4.4 Policy Iteration

📖 **Referans:** Sutton & Barto, Sayfa 80-81, Section 4.3

> *"Once a policy, π, has been improved using $v_\pi$ to yield a better policy, π', we can then compute $v_{\pi'}$ and improve it again to yield an even better π''."* (s. 80)

**Policy Evaluation** ve **Policy Improvement**'ı sırayla uygula (s. 80):

$$\pi_0 \xrightarrow{E} v_{\pi_0} \xrightarrow{I} \pi_1 \xrightarrow{E} v_{\pi_1} \xrightarrow{I} \pi_2 \xrightarrow{E} ... \xrightarrow{I} \pi_* \xrightarrow{E} v_*$$

> *"This way of finding an optimal policy is called policy iteration."* (s. 80)

### Convergence (s. 80)
> *"A finite MDP has only a finite number of policies... so this process must converge to an optimal policy and optimal value function in a finite number of iterations."*

In [ ]:
def policy_iteration(env, gamma=1.0, theta=1e-8):
    """
    Policy Iteration algoritması.
    Referans: Sutton & Barto, Algorithm (s. 80)
    
    "1. Initialization... 2. Policy Evaluation... 3. Policy Improvement..."
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        history: Iteration history
    """
    # "1. Initialization: V(s) ∈ R and π(s) ∈ A(s) arbitrarily"
    policy = np.random.randint(0, env.n_actions, size=env.n_states)
    
    history = []
    iteration = 0
    
    while True:
        # "2. Policy Evaluation" (s. 80)
        V, _ = policy_evaluation(env, policy, gamma, theta)
        
        # "3. Policy Improvement" (s. 80)
        # "policy-stable ← true"
        new_policy = policy_improvement(env, V, gamma)
        
        history.append({
            'iteration': iteration,
            'policy': policy.copy(),
            'V': V.copy()
        })
        
        # "If policy-stable, then stop" (s. 80)
        if np.array_equal(policy, new_policy):
            print(f"Policy Iteration converged in {iteration + 1} iterations")
            break
        
        policy = new_policy
        iteration += 1
    
    return policy, V, history

optimal_policy, V_star, pi_history = policy_iteration(env)

In [ ]:
# Sonuçları görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Value function
ax = axes[0]
V_grid = V_star.reshape(env.rows, env.cols)
im = ax.imshow(V_grid, cmap='RdYlGn')
for i in range(env.rows):
    for j in range(env.cols):
        color = 'white' if abs(V_grid[i, j]) > 1 else 'black'
        ax.text(j, i, f'{V_grid[i, j]:.1f}', ha='center', va='center',
               fontsize=16, color=color, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Optimal V*(s)', fontsize=14)

# Policy
ax = axes[1]
for s in range(env.n_states):
    row, col = env.state_to_pos(s)
    color = 'lightgreen' if s in env.terminal_states else 'lightblue'
    rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    
    if s not in env.terminal_states:
        ax.text(col + 0.5, env.rows - row - 0.5, env.action_symbols[optimal_policy[s]],
               ha='center', va='center', fontsize=28, fontweight='bold')
    else:
        ax.text(col + 0.5, env.rows - row - 0.5, 'T',
               ha='center', va='center', fontsize=20, fontweight='bold')

ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Optimal Policy π*', fontsize=14)

plt.tight_layout()
plt.show()

---
## 4.5 Value Iteration

📖 **Referans:** Sutton & Barto, Sayfa 82-84, Section 4.4

> *"One drawback to policy iteration is that each of its iterations involves policy evaluation, which may itself be a protracted iterative computation requiring multiple sweeps through the state set."* (s. 82)

Policy Iteration'da her seferinde **tam** policy evaluation yapmak yerine, sadece **bir sweep** yap.

### Value Iteration Update - Equation 4.10 (s. 83)

$$v_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_k(s')]$$

> *"Value iteration effectively combines, in each of its sweeps, one sweep of policy evaluation and one sweep of policy improvement."* (s. 83)

### Convergence (s. 83)
> *"Value iteration is obtained simply by turning the Bellman optimality equation into an update rule... value iteration converges to v*"*

In [ ]:
def value_iteration(env, gamma=1.0, theta=1e-8):
    """
    Value Iteration algoritması.
    Referans: Sutton & Barto, Algorithm (s. 83)
    
    "Initialize V(s)... Repeat... until Δ < θ"
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        history: Value function at each iteration
    """
    # "Initialize V(s), for all s ∈ S+, arbitrarily except that V(terminal) = 0"
    V = np.zeros(env.n_states)
    history = [V.copy()]
    
    iteration = 0
    # "Repeat"
    while True:
        delta = 0  # "Δ ← 0"
        
        # "For each s ∈ S:"
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]  # "v ← V(s)"
            
            # "V(s) ← max_a Σ_{s',r} p(s',r|s,a)[r + γV(s')]" - Eq 4.10
            q_values = np.zeros(env.n_actions)
            for a in range(env.n_actions):
                for prob, next_s, reward, done in env.get_transitions(s, a):
                    q_values[a] += prob * (reward + gamma * V[next_s])
            
            V[s] = np.max(q_values)
            delta = max(delta, abs(v - V[s]))  # "Δ ← max(Δ, |v − V(s)|)"
        
        iteration += 1
        history.append(V.copy())
        
        # "until Δ < θ"
        if delta < theta:
            break
    
    # "Output a deterministic policy, π ≈ π*" - greedy policy extraction
    policy = policy_improvement(env, V, gamma)
    
    print(f"Value Iteration converged in {iteration} iterations")
    return policy, V, history

vi_policy, vi_V, vi_history = value_iteration(env)

In [ ]:
# Value Iteration convergence
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

iterations_to_show = [0, 1, 2, 3, 5, len(vi_history)-1]

for idx, (ax, it) in enumerate(zip(axes.flat, iterations_to_show)):
    V_it = vi_history[it].reshape(env.rows, env.cols)
    im = ax.imshow(V_it, cmap='RdYlGn', vmin=-14, vmax=0)
    
    for i in range(env.rows):
        for j in range(env.cols):
            ax.text(j, i, f'{V_it[i, j]:.1f}', ha='center', va='center',
                   fontsize=12, fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Iteration {it}', fontsize=12)

plt.suptitle('Value Iteration: Convergence', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4.6 Policy Iteration vs Value Iteration

📖 **Referans:** Sutton & Barto, Sayfa 82-84

> *"In fact, the policy evaluation step of policy iteration can be truncated in several ways without losing the convergence guarantees of policy iteration."* (s. 82)

| Özellik | Policy Iteration | Value Iteration |
|---------|-----------------|----------------|
| Her iterasyon | Full evaluation + 1 improvement | 1 Bellman optimality update |
| İterasyon sayısı | Az | Çok |
| İterasyon maliyeti | Yüksek | Düşük |
| Toplam maliyet | Genelde benzer | Genelde benzer |

> *"In practice, value iteration terminates once the value function changes by only a small amount in a sweep."* (s. 83)

In [ ]:
# Karşılaştırma
print("Policy Iteration vs Value Iteration")
print("="*40)
print(f"\nPolicy Iteration: {len(pi_history)} policy iterations")
print(f"Value Iteration: {len(vi_history)-1} value iterations")

# Aynı sonuca ulaştıklarını doğrula
print(f"\nSame optimal policy: {np.array_equal(optimal_policy, vi_policy)}")
print(f"Same optimal V (approx): {np.allclose(V_star, vi_V)}")

---
## 4.7 Gambler's Problem

📖 **Referans:** Sutton & Barto, Example 4.3 (Sayfa 84-85)

> *"A gambler has the opportunity to make bets on the outcomes of a sequence of coin flips. If the coin comes up heads, he wins as many dollars as he has staked on that flip; if it is tails, he loses his stake."* (s. 84)

Klasik bir DP örneği:
- Kumarbaz $p_h$ olasılıkla yazı gelen bir para atıyor
- 100\$ kazanırsa oyun biter (kazandı)
- 0\$ olursa oyun biter (kaybetti)
- Her turda, mevcut parasının bir kısmını bahis olarak koyar

### MDP Formulation (s. 84)
- **State:** Kumarbazın mevcut sermayesi $s \in \{1, 2, ..., 99\}$
- **Actions:** Stakes $a \in \{0, 1, ..., \min(s, 100-s)\}$
- **Reward:** Goal'e ulaşınca +1, diğer türlü 0

> *"Figure 4.3 shows the change in the value function over successive sweeps of value iteration, and the final policy found"* (s. 85)

In [ ]:
def gamblers_problem(p_heads=0.4, gamma=1.0, theta=1e-9):
    """
    Gambler's Problem with Value Iteration.
    Referans: Example 4.3 (s. 84-85)
    
    "The state is the gambler's capital, s ∈ {1, 2, . . . , 99}. 
    The actions are stakes, a ∈ {0, 1, . . . , min(s, 100−s)}"
    """
    
    goal = 100
    # "V(s) arbitrarily, for all s ∈ S+, except V(0) = V(100) = 0"
    V = np.zeros(goal + 1)
    V[goal] = 1.0  # "Reward is +1 if the gambler reaches his goal" (s. 84)
    
    history = [V.copy()]
    
    while True:
        delta = 0
        
        for s in range(1, goal):
            v = V[s]
            
            # "Stakes, a ∈ {0, 1, . . . , min(s, 100−s)}"
            max_stake = min(s, goal - s)
            action_values = []
            
            for stake in range(1, max_stake + 1):
                # Heads: win (s + stake), Tails: lose (s - stake)
                # "p_h is the probability of the coin coming up heads"
                value = p_heads * V[s + stake] + (1 - p_heads) * V[s - stake]
                action_values.append(value)
            
            V[s] = max(action_values)
            delta = max(delta, abs(v - V[s]))
        
        history.append(V.copy())
        
        if delta < theta:
            break
    
    # Extract policy
    policy = np.zeros(goal + 1, dtype=int)
    for s in range(1, goal):
        max_stake = min(s, goal - s)
        action_values = []
        
        for stake in range(1, max_stake + 1):
            value = p_heads * V[s + stake] + (1 - p_heads) * V[s - stake]
            action_values.append(value)
        
        # "deterministic optimal policy" - en küçük stake'i tercih et
        policy[s] = np.argmax(action_values) + 1
    
    return V, policy, history

V_gambler, policy_gambler, _ = gamblers_problem(p_heads=0.4)
print("Gambler's Problem solved (p_h=0.4)")

In [ ]:
# Figure 4.3 (s. 85) - Gambler's Problem Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Value function
axes[0].plot(V_gambler, 'b-', linewidth=2)
axes[0].set_xlabel('Capital ($)')
axes[0].set_ylabel('Value V(s)')
axes[0].set_title("Gambler's Problem: Value Function (p_h=0.4)\nFigure 4.3 (s. 85)")
axes[0].grid(True, alpha=0.3)

# Policy - "Final policy (showing only those states for which the optimal action is not trivially determined)"
axes[1].bar(range(len(policy_gambler)), policy_gambler, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Capital ($)')
axes[1].set_ylabel('Stake ($)')
axes[1].set_title("Gambler's Problem: Optimal Policy (p_h=0.4)\nFigure 4.3 (s. 85)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Özet

📖 **Chapter 4 Key Points (s. 73-92)**

| Algoritma | Sayfa | Kullanım | Gereksinim |
|-----------|-------|----------|------------|
| **Policy Evaluation** | s. 74-76 | V^π hesapla | Policy + Model |
| **Policy Improvement** | s. 76-78 | π'den daha iyi π' bul | V^π + Model |
| **Policy Iteration** | s. 80-81 | Optimal π* bul | Model |
| **Value Iteration** | s. 82-84 | Optimal V* bul | Model |

### Anahtar Denklemler

**Iterative Policy Evaluation** (Eq. 4.5, s. 75):
$$v_{k+1}(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma v_k(s')]$$

**Policy Improvement** (Eq. 4.9, s. 78):
$$\pi'(s) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_\pi(s')]$$

**Value Iteration** (Eq. 4.10, s. 83):
$$v_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_k(s')]$$

### Önemli Noktalar (s. 86-87)
> *"DP may not be practical for very large problems, but compared with other methods for solving MDPs, DP methods are actually quite efficient."*

- DP **tam model** gerektirir
- Her iki algoritma da **optimal**'e yakınsar
- Büyük state space'lerde **impractical** olabilir

---
### Sonraki Notebook
**05 - Monte Carlo Methods** *(Chapter 5, s. 91-113)*: Model-free prediction and control